# Rung 2 - grokking audit at P=113 (Kaggle or Colab GPU)

Two jobs, one session, decided **before** seeing results:

| Job | Protocol | What it answers |
|-----|----------|-----------------|
| A | `--protocol nanda`, 40,000 full-batch steps, seeds 0,1,2 | **Positive control.** Does the original Nanda et al. recipe grok in this codebase? |
| B | `--protocol repo`, the exact old run (batch 512, 5000 epochs, seeds 0,1,2) | **Re-judges the NO-GROK verdict** with the energy metric instead of the L1 `k_99`. |

**Falsifier for A:** if val accuracy is below 0.99 or `energy_sparse` is 0 on 2 or more of
3 seeds, there is a bug in the model or the training loop - debug it, do not sweep.
Expected for a working setup: val acc ~1.0 after roughly 10k steps with about 5 key frequencies
(`k_energy_90` around 5 of 56).

**Before running:** enable a GPU (Kaggle: Settings -> Accelerator -> GPU T4, and Internet On;
Colab: Runtime -> Change runtime type -> GPU). Set `BRANCH` below to a branch that is pushed.


In [ ]:
BRANCH = "dev-v1"  # must exist on GitHub
!git clone --branch $BRANCH --depth 1 https://github.com/AlessioBrillo/from-gradient-to-transformer.git repo
%cd repo
!git rev-parse --short HEAD

In [ ]:
!pip install -q uv
!uv sync --frozen
!uv run python -c "import torch; assert torch.cuda.is_available(), 'no GPU'; print(torch.__version__, torch.cuda.get_device_name(0))"

In [ ]:
# Job A - positive control (Nanda protocol). Resumable: re-run this cell after a disconnect.
!uv run python -m src.experiments.exp2_grokking --protocol nanda --modulus 113 --epochs 40000 --seeds 0,1,2 --checkpoint-dir checkpoints/nanda --checkpoint-every 5000 --resume --progress-interval 100 --manifest-path results/grokking_nanda_p113.json

In [ ]:
# Job B - the old run, unchanged, now also reporting the energy metric.
!uv run python -m src.experiments.exp2_grokking --protocol repo --modulus 113 --epochs 5000 --batch-size 512 --seeds 0,1,2 --checkpoint-dir checkpoints/repo --checkpoint-every 1000 --resume --manifest-path results/grokking_repo_p113_energy.json

In [ ]:
import json
for name in ('grokking_nanda_p113', 'grokking_repo_p113_energy'):
    m = json.load(open(f'results/{name}.json'))
    a = m['aggregate']
    print(name, 'device', m['device'], 'sha', m['git_sha'], 'dirty', m['git_dirty'])
    for k in ('final_val_acc', 'generalization_epoch', 'k_energy_90', 'frac_energy_top5', 'energy_sparse', 'k_99_percent'):
        print(f'  {k:<22}{a[k]["mean"]:10.3f} +- {a[k]["std"]:.3f}  (min {a[k]["min"]:.3f}, max {a[k]["max"]:.3f})')

In [ ]:
!zip -j grokking_p113_results.zip results/grokking_nanda_p113.json results/grokking_repo_p113_energy.json
try:
    from google.colab import files
    files.download('grokking_p113_results.zip')
except ImportError:
    print('Kaggle: the zip is in the notebook output (/kaggle/working).')

## After downloading

1. Copy both JSON files into the local clone's `results/`, commit them on a branch.
2. Do not edit `portfolio/RESULTS.md` or the paper by hand: cite the manifests with `<!-- manifest: results/grokking_nanda_p113.json -->` and run `make verify-claims`.
3. Report the outcome exactly as it is, including the case where A does not grok.